In [1]:
!pip install optuna-integration

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 4.3 MB/s eta 0:00:00


In [2]:
import os
import warnings
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import lightgbm as lgb
import optuna
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import CountVectorizer

In [3]:
def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    
    warnings.filterwarnings("ignore")
    random.seed(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(False)
    
    print(f"Random seed set to {seed}")

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2 ** 32
    random.seed(worker_seed)
    np.random.seed(worker_seed)

In [4]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

data_generator = torch.Generator()
data_generator.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Random seed set to 42
Device: cuda


In [5]:
INPUT_ROOT = "/kaggle/input/datasets/minhnguyn132/input-df26-v3"
WORK_DIR = "/kaggle/working"

X_TRAIN = f"{INPUT_ROOT}/X_train.csv"
X_VAL = f"{INPUT_ROOT}/X_val.csv"
X_TEST = f"{INPUT_ROOT}/X_test.csv"
Y_TRAIN = f"{INPUT_ROOT}/Y_train.csv"
Y_VAL = f"{INPUT_ROOT}/Y_val.csv"

SUBMISSION_FILE = f"{WORK_DIR}/submission.csv"
CHECKPOINT_FILE = "/kaggle/input/datasets/minhnguyn132/input-df26-v3/best_model_gnn_retrained.pth"

In [6]:
SEQ_LENGTH = 66
FEATURE_COLS = [f"feature_{i}" for i in range(1, SEQ_LENGTH + 1)]

ATTRIBUTE_LIST = [1, 2, 3, 4, 5, 6]
ATTRIBUTE_COLS = ["start_year", "attr_1", "attr_2", "attr_3",
                  "end_year", "attr_4", "attr_5", "attr_6"]

EMBEDDING_DIM = 256
DILATIONS = [1, 2, 4, 8]
DROPOUT_RATE = 0.3
HEADS = 4
WINDOW_SIZE = 3

BATCH_SIZE = 1024

M_LIST_METRIC = [12.0, 31.0, 99.0, 12.0, 31.0, 99.0]
W_LIST_VALS = [1.0, 1.0, 100.0, 1.0, 1.0, 100.0]

DAYS_IN_MONTH = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
DAYS_MAP = {
    1: 31, 2: 28, 3: 31, 4: 30, 5: 31, 6: 30,
    7: 31, 8: 31, 9: 30, 10: 31, 11: 30, 12: 31
}

In [7]:
def drop_duplicates_custom(x_df, y_df, feature_cols=FEATURE_COLS, name="Train"):
    len_before = len(x_df)
    keep_idx = x_df.drop_duplicates(subset=feature_cols, keep="first").index
    
    x_df = x_df.loc[keep_idx].reset_index(drop=True)
    y_df = y_df.loc[keep_idx].reset_index(drop=True)
    
    print(f" - Loại bỏ {len_before - len(x_df)} hàng trùng lặp trong {name}")
    return x_df, y_df

In [8]:
def drop_overlap_train_val(x_train, y_train, x_val, feature_cols=FEATURE_COLS):
    len_before = len(x_train)
    
    val_features_unique = x_val[feature_cols].drop_duplicates().copy()
    val_features_unique["is_in_val"] = True
    
    x_train["old_idx"] = x_train.index
    train_merged = x_train.merge(val_features_unique, on=feature_cols, how="left")
    
    clean_train_idx = train_merged[train_merged["is_in_val"].isna()]["old_idx"]
    
    x_train = x_train.loc[clean_train_idx].drop(columns=["old_idx"]).reset_index(drop=True)
    y_train = y_train.loc[clean_train_idx].reset_index(drop=True)
    
    print(f" - Loại bỏ {len_before - len(x_train)} chuỗi Train bị trùng với Val")
    return x_train, y_train

In [9]:
def validate_and_clean_dates(x_df, y_df, days_map=DAYS_MAP, name="Train"):
    max_days_start = y_df["attr_1"].map(days_map)
    invalid_start_day_mask = y_df["attr_2"] > max_days_start

    max_days_end = y_df["attr_4"].map(days_map)
    invalid_end_day_mask = y_df["attr_5"] > max_days_end

    all_invalid_mask = invalid_start_day_mask | invalid_end_day_mask
    invalid_count = all_invalid_mask.sum()

    if invalid_count > 0:
        print(f" - Phát hiện {invalid_count} hàng có lỗi ngày trong {name}")
        
        y_df.loc[invalid_start_day_mask, "attr_2"] = max_days_start[invalid_start_day_mask]
        y_df.loc[invalid_end_day_mask, "attr_5"] = max_days_end[invalid_end_day_mask]
        
    else:
        print(f" - Không có lỗi ngày tháng trong {name}")

    return x_df, y_df

In [10]:
def get_vocab_size(x_train, x_val, x_test, feature_cols=FEATURE_COLS):
    all_x_data = pd.concat([x_train[feature_cols], x_val[feature_cols]], axis=0)
    unique_ids = pd.unique(all_x_data.values.ravel())
    unique_ids = [uid for uid in unique_ids if not pd.isna(uid) and uid != 0]
    
    id_to_index = {0: 0, "UNK": 1}
    for idx, raw_id in enumerate(sorted(unique_ids), start=2):
        id_to_index[raw_id] = idx
        
    vocab_size = max(id_to_index.values()) + 1
    print(f" - Kích thước từ điển {vocab_size} (PAD=0, UNK=1)")

    map_func = np.vectorize(lambda x: id_to_index.get(x, 1))
    
    for df in [x_train, x_val, x_test]:
        df.loc[:, feature_cols] = map_func(df[feature_cols].values)

    return x_train, x_val, x_test, vocab_size

In [11]:
def process_data(x_train, y_train, x_val, y_val, x_test,
                 feature_cols=FEATURE_COLS,
                 attr_cols=ATTRIBUTE_COLS,
                 days_map=DAYS_MAP):
    
    for df in [x_train, x_val, x_test]:
        df[feature_cols] = df[feature_cols].fillna(0).astype(np.int64)
    for df in [y_train, y_val]:
        df["start_year"] = 0
        df["end_year"] = (df["attr_4"] < df["attr_1"]).astype(np.int64)
        df[attr_cols] = df[attr_cols].fillna(0).astype(np.float32)

    print("Xử lý trùng lặp nội bộ...")
    x_train, y_train = drop_duplicates_custom(x_train, y_train, feature_cols, "Train")
    x_val, y_val = drop_duplicates_custom(x_val, y_val, feature_cols, "Val")
    
    print("Xử lý trùng lặp giữa Train và Val...")
    x_train, y_train = drop_overlap_train_val(x_train, y_train, x_val, feature_cols)

    print("Xử lý lỗi ngày tháng...")
    x_train, y_train = validate_and_clean_dates(x_train, y_train, days_map, "Train")
    x_val, y_val = validate_and_clean_dates(x_val, y_val, days_map, "Val")

    print("Xây dựng từ điển và map index...")
    x_train, x_val, x_test, vocab_size = get_vocab_size(x_train, x_val, x_test, feature_cols)
    
    return x_train, y_train, x_val, y_val, x_test, vocab_size

In [12]:
class UserBehaviorDataset(Dataset):
    def __init__(self, x_df, y_df=None, 
                 feature_cols=FEATURE_COLS, attr_cols=ATTRIBUTE_COLS, augment=False):
        self.x_data = x_df[feature_cols].values
        self.augment = augment 
        
        self.has_labels = y_df is not None
        if self.has_labels:
            self.y_data = y_df[attr_cols].values

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        x_tensor = torch.tensor(self.x_data[idx], dtype=torch.long)
        
        if self.augment:
            actions = x_tensor[x_tensor != 0]
            n_actions = len(actions)
            total_slots = len(x_tensor) 
            
            if torch.rand(1).item() < 0.5:
                n_mask = (n_actions // 10) + 1
                possible_indices = torch.arange(n_actions - 1)
                
                if len(possible_indices) >= n_mask:
                    mask_indices = possible_indices[torch.randperm(len(possible_indices))[:n_mask]]
                    actions[mask_indices] = 0
            
            if torch.rand(1).item() < 0.5:
                new_indices = torch.randperm(total_slots)[:n_actions].sort()[0]
                dilated_x = torch.zeros_like(x_tensor)
                dilated_x[new_indices] = actions
                x_tensor = dilated_x
            else:
                new_x = torch.zeros_like(x_tensor)
                new_x[:n_actions] = actions
                x_tensor = new_x

        if self.has_labels:
            y_tensor = torch.tensor(self.y_data[idx], dtype=torch.float32)
            return x_tensor, y_tensor
        
        return x_tensor

In [13]:
def create_dataloaders(x_train, y_train, x_val, y_val, x_test, 
                       batch_size=BATCH_SIZE,
                       num_workers=2,
                       seed_worker=seed_worker,
                       data_generator=data_generator):
    
    train_dataset = UserBehaviorDataset(x_train, y_train, augment=False)
    val_dataset = UserBehaviorDataset(x_val, y_val, augment=False)
    test_dataset = UserBehaviorDataset(x_test, None, augment=False)

    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers,           
        pin_memory=True,
        persistent_workers=False 
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers,           
        pin_memory=True,
        persistent_workers=False 
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers,
        pin_memory=True
    )
    
    return train_loader, val_loader, test_loader

In [14]:
class DenseGAT(nn.Module):
    def __init__(self, in_channels, out_channels, heads=4, dropout=0.2):
        super().__init__()
        self.heads, self.out_channels, self.head_dim = heads, out_channels, out_channels // heads
        self.lin = nn.Linear(in_channels, out_channels, bias=False)
        self.att_src = nn.Parameter(torch.Tensor(1, heads, 1, self.head_dim))
        self.att_dst = nn.Parameter(torch.Tensor(1, heads, 1, self.head_dim))
        self.leaky_relu = nn.LeakyReLU(0.2)
        self.dropout = nn.Dropout(dropout)
        
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att_src)
        nn.init.xavier_uniform_(self.att_dst)

    def forward(self, x, adj_mask):
        B, L, _ = x.size()
        h = self.lin(x).view(B, L, self.heads, self.head_dim).transpose(1, 2) 
        alpha_src = (h * self.att_src).sum(dim=-1, keepdim=True)
        alpha_dst = (h * self.att_dst).sum(dim=-1, keepdim=True)
        
        e = self.leaky_relu(alpha_src + alpha_dst.transpose(2, 3))
        fill_value = torch.finfo(e.dtype).min
        e = e.masked_fill(~adj_mask.unsqueeze(1), fill_value)
        
        alpha = self.dropout(F.softmax(e, dim=-1))
        return torch.matmul(alpha, h).transpose(1, 2).contiguous().view(B, L, self.out_channels)

class GatedDirectedGATLayer(nn.Module):
    def __init__(self, d_model, heads=4, dropout=0.2):
        super().__init__()
        self.gat_past = DenseGAT(d_model, d_model, heads, dropout)
        self.gat_future = DenseGAT(d_model, d_model, heads, dropout)
        self.msg_proj = nn.Linear(d_model * 2, d_model)
        self.gru_cell = nn.GRUCell(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, adj_past, adj_future, mask):
        B, L, C = x.size()
        m_comb = F.relu(self.msg_proj(torch.cat([self.gat_past(x, adj_past), self.gat_future(x, adj_future)], dim=-1)))
        h_new = self.gru_cell(m_comb.view(-1, C), x.view(-1, C)).view(B, L, C)
        return self.norm(h_new) * mask.unsqueeze(-1).float()

class GraphormerLightLayer(nn.Module):
    def __init__(self, d_model, heads=4, dropout=0.2):
        super().__init__()
        self.gnn = GatedDirectedGATLayer(d_model, heads, dropout)
        self.self_attn = nn.MultiheadAttention(d_model, heads, dropout=dropout, batch_first=True)
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model*2), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model*2, d_model))

    def forward(self, x, adj_past, adj_future, mask):
        x = self.norm1(x + self.gnn(x, adj_past, adj_future, mask))
        attn_out, _ = self.self_attn(x, x, x, key_padding_mask=~mask)
        x = self.norm2(x + attn_out)
        return x + self.ffn(x)

class GCE_FusionLayer(nn.Module):
    def __init__(self, vocab_size, d_model, dropout=0.2):
        super().__init__()
        self.global_memory = nn.Embedding(vocab_size, d_model, padding_idx=0)
        
        self.fusion_gate = nn.Linear(d_model * 2, d_model)
        
        self.transform = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)
        
        nn.init.xavier_normal_(self.fusion_gate.weight)
        nn.init.xavier_normal_(self.transform.weight)

    def forward(self, local_features, input_ids):
        global_features = self.global_memory(input_ids) 
        
        concat_features = torch.cat([local_features, global_features], dim=-1)
        gate_score = torch.sigmoid(self.fusion_gate(concat_features)) 
        
        fused_features = gate_score * local_features + (1.0 - gate_score) * global_features
        
        out = self.dropout(F.gelu(self.transform(fused_features)))
        return self.norm(local_features + out) 
        
class AttentionPooling1D(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.attn = nn.Linear(in_features, 1)

    def forward(self, x, mask):
        scores = self.attn(x) 
        
        if mask is not None:
            fill_value = torch.finfo(scores.dtype).min
            scores = scores.masked_fill(~mask.unsqueeze(-1), fill_value)
            
        attn_weights = F.softmax(scores, dim=1)
        return torch.sum(x * attn_weights, dim=1)

class CascadeRegressionHead(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.head_start = nn.Sequential(
            nn.Linear(d_model, 64), 
            nn.GELU(), 
            nn.Linear(64, 3)
        )
        
        self.head_factory = nn.Sequential(
            nn.Linear(d_model, 128), 
            nn.LayerNorm(128), 
            nn.GELU(), 
            nn.Dropout(0.2), 
            nn.Linear(128, 64), 
            nn.GELU(), 
            nn.Linear(64, 2)
        )
        
        self.start_proj = nn.Sequential(
            nn.Linear(3, 16),
            nn.GELU(),
            nn.Linear(16, 32)
        )
        
        self.head_end = nn.Sequential(
            nn.Linear(d_model + 32, 64), 
            nn.GELU(), 
            nn.Linear(64, 3)
        )

    def forward(self, x):
        start_preds = self.head_start(x)  
        
        factory_preds = self.head_factory(x) 
        
        start_context = self.start_proj(start_preds)
        end_in = torch.cat([x, start_context], dim=-1)
        end_preds = self.head_end(end_in)    
        
        s_year = torch.sigmoid(start_preds[:, 0])
        e_year = torch.sigmoid(end_preds[:, 0])
        
        s_month = 1.0 + 11.0 * torch.sigmoid(start_preds[:, 1])
        e_month = 1.0 + 11.0 * torch.sigmoid(end_preds[:, 1])
        
        s_day = 1.0 + 30.0 * torch.sigmoid(start_preds[:, 2])
        e_day = 1.0 + 30.0 * torch.sigmoid(end_preds[:, 2])
        
        s_power = 99.0 * torch.sigmoid(factory_preds[:, 0])
        e_power = 99.0 * torch.sigmoid(factory_preds[:, 1])
        
        return [
            s_year, s_month, s_day, s_power,
            e_year, e_month, e_day, e_power
        ]
        
class AdvancedTAGNet(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.window = WINDOW_SIZE
        self.dilations = DILATIONS
        
        self.embedding = nn.Embedding(vocab_size, EMBEDDING_DIM, padding_idx=0)
        self.pos_embedding = nn.Embedding(SEQ_LENGTH, EMBEDDING_DIM)
        
        self.layers = nn.ModuleList([
            GraphormerLightLayer(EMBEDDING_DIM, HEADS, DROPOUT_RATE) 
            for _ in DILATIONS
        ])
        
        self.gce_fusion = GCE_FusionLayer(vocab_size, EMBEDDING_DIM, DROPOUT_RATE)
        
        self.attn_pool = AttentionPooling1D(EMBEDDING_DIM)
        self.dropout = nn.Dropout(DROPOUT_RATE)
        
        self.cascade_head = CascadeRegressionHead(EMBEDDING_DIM)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_normal_(m.weight)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02) 
        elif isinstance(m, nn.GRUCell):
            for name, param in m.named_parameters():
                if 'weight' in name:
                    nn.init.orthogonal_(param)
                elif 'bias' in name:
                    nn.init.constant_(param, 0)
                
    def _get_adj(self, pos, valid, win, dil, dev):
        diff = torch.abs(pos.unsqueeze(2) - pos.unsqueeze(1))
        in_win = (diff <= win * dil) & (diff % dil == 0)
        eye = torch.eye(pos.size(1), dtype=torch.bool, device=dev).unsqueeze(0)
        return (valid & (in_win & (pos.unsqueeze(2) - pos.unsqueeze(1) < 0))) | eye, \
               (valid & (in_win & (pos.unsqueeze(2) - pos.unsqueeze(1) > 0))) | eye

    def forward(self, x):
        B, L = x.size()
        mask = (x != 0)
        pos = torch.arange(L, device=x.device).unsqueeze(0).expand(B, L)
        
        h = (self.embedding(x) + self.pos_embedding(pos)) * mask.unsqueeze(-1).float()
        log_pos = (torch.cumsum(mask.long(), dim=1) - 1).clamp(min=0)
        valid = mask.unsqueeze(1) & mask.unsqueeze(2)
        
        for i, layer in enumerate(self.layers):
            adj_p, adj_f = self._get_adj(log_pos, valid, self.window, self.dilations[i], x.device)
            h = layer(h, adj_p, adj_f, mask)
            
        h = self.gce_fusion(h, x)
            
        x_drop = self.dropout(self.attn_pool(h, mask))
        return self.cascade_head(x_drop)

In [15]:
def post_process_predictions(preds_tensor):
    preds = preds_tensor.clone()
    preds = torch.round(preds)
    
    preds[:, [0, 3]] = (preds[:, [0, 3]] - 1) % 12 + 1
    preds[:, [0, 3]] = torch.clamp(preds[:, [0, 3]], 1, 12)

    days_in_month_t = torch.tensor(DAYS_IN_MONTH, device=preds.device, dtype=preds.dtype)
    
    for col_idx, month_idx in [(1, 0), (4, 3)]:
        m_indices = (preds[:, month_idx].long() - 1).clamp(0, 11)
        max_days = days_in_month_t[m_indices]
        
        preds[:, col_idx] = torch.clamp(preds[:, col_idx], min=1)
        preds[:, col_idx] = torch.min(preds[:, col_idx], max_days)

    preds[:, [2, 5]] = torch.clamp(preds[:, [2, 5]], 0, 99)
    
    return preds

In [16]:
def extract_hybrid_features(model, dataloader, device=DEVICE):
    model.eval()
    
    extracted_embeddings = []
    
    def hook_fn(module, input, output):
        extracted_embeddings.append(output.detach().cpu().numpy())
        
    handle = model.attn_pool.register_forward_hook(hook_fn)
    
    all_preds = []
    y_true_list = []
    
    print("Đang trích xuất đặc trưng...")
    with torch.no_grad():
        for batch in dataloader:
            if isinstance(batch, (list, tuple)) and len(batch) == 2:
                x, y = batch
                y_true_list.append(y.cpu().numpy())
            else:
                x = batch[0] if isinstance(batch, (list, tuple)) else batch
                
            x = x.to(device)
            
            outputs = model(x) 
            
            preds_batch = torch.stack(outputs, dim=1).cpu().numpy()
            all_preds.append(preds_batch)
            
    handle.remove()
    
    final_embeddings = np.concatenate(extracted_embeddings, axis=0)
    final_preds = np.concatenate(all_preds, axis=0)
    
    x_hybrid = np.concatenate([final_embeddings, final_preds], axis=1) 
    
    if len(y_true_list) > 0:
        y_true = np.concatenate(y_true_list, axis=0)
        return x_hybrid, y_true
    else:
        return x_hybrid

In [17]:
class SequenceFeatureExtractor:
    def __init__(self, top_k_vocab=50):
        self.vectorizer = CountVectorizer(max_features=top_k_vocab, token_pattern=r"(?u)\b\w+\b")
        self.is_fitted = False

    def extract(self, x_df, feature_cols=FEATURE_COLS):
        print(f"Đang trích xuất đặc cho {len(x_df)} mẫu...")
        seqs = x_df[feature_cols].values 
        
        seq_lengths = np.sum(seqs != 0, axis=1)
        consecutive_repeats = np.sum((seqs[:, 1:] == seqs[:, :-1]) & (seqs[:, 1:] != 0), axis=1)
        padding_ratio = (len(feature_cols) - seq_lengths) / len(feature_cols)
        
        num_uniques = np.zeros(len(seqs), dtype=np.float32)
        diversity = np.zeros(len(seqs), dtype=np.float32)
        first_codes = np.zeros(len(seqs), dtype=np.float32)
        last_codes = np.zeros(len(seqs), dtype=np.float32)
        
        most_frequent_codes = np.zeros(len(seqs), dtype=np.float32)
        obsession_ratios = np.zeros(len(seqs), dtype=np.float32)
        center_of_mass = np.zeros(len(seqs), dtype=np.float32)
        
        seq_strings = [] 
        for i, row in enumerate(seqs):
            non_zeros = row[row != 0]
            if len(non_zeros) > 0:
                vals, counts = np.unique(non_zeros, return_counts=True)
                num_uniques[i] = len(vals)
                diversity[i] = num_uniques[i] / len(non_zeros)
                first_codes[i] = non_zeros[0]
                last_codes[i] = non_zeros[-1] 
                
                best_idx = np.argmax(counts)
                most_frequent_codes[i] = vals[best_idx]
                obsession_ratios[i] = counts[best_idx] / len(non_zeros)
                
                center_of_mass[i] = np.mean(np.arange(len(non_zeros))) / len(non_zeros)
                
                seq_strings.append(" ".join(non_zeros.astype(str)))
            else:
                seq_strings.append("")

        if not self.is_fitted:
            bow_features = self.vectorizer.fit_transform(seq_strings).toarray()
            self.is_fitted = True
        else:
            bow_features = self.vectorizer.transform(seq_strings).toarray()
        
        manual_features = np.column_stack([
            seq_lengths,
            num_uniques,
            diversity,
            first_codes,
            last_codes,
            consecutive_repeats,
            padding_ratio,
            most_frequent_codes,
            obsession_ratios,
            center_of_mass,
            bow_features  
        ])
        
        return manual_features.astype(np.float32)

In [18]:
x_train = pd.read_csv(X_TRAIN)
y_train = pd.read_csv(Y_TRAIN)
x_val = pd.read_csv(X_VAL)
y_val = pd.read_csv(Y_VAL)
x_test = pd.read_csv(X_TEST)

x_train, y_train, x_val, y_val, x_test, VOCAB_SIZE = process_data(
    x_train, y_train, x_val, y_val, x_test
)

y_val_orig = (y_val[ATTRIBUTE_COLS].values)[:, [1, 2, 3, 5, 6, 7]] 

train_loader, val_loader, test_loader = create_dataloaders(
    x_train, y_train, x_val, y_val, x_test
)

Xử lý trùng lặp nội bộ...
 - Loại bỏ 99 hàng trùng lặp trong Train
 - Loại bỏ 110 hàng trùng lặp trong Val
Xử lý trùng lặp giữa Train và Val...
 - Loại bỏ 187 chuỗi Train bị trùng với Val
Xử lý lỗi ngày tháng...
 - Phát hiện 1010 hàng có lỗi ngày trong Train
 - Phát hiện 749 hàng có lỗi ngày trong Val
Xây dựng từ điển và map index...
 - Kích thước từ điển 934 (PAD=0, UNK=1)


In [19]:
extractor = SequenceFeatureExtractor(top_k_vocab=50)

x_train_manual = extractor.extract(x_train) 
x_val_manual = extractor.extract(x_val)
x_test_manual = extractor.extract(x_test)

Đang trích xuất đặc cho 54714 mẫu...
Đang trích xuất đặc cho 39890 mẫu...
Đang trích xuất đặc cho 108673 mẫu...


In [20]:
model = AdvancedTAGNet(vocab_size=VOCAB_SIZE).to(DEVICE)
model.load_state_dict(torch.load(CHECKPOINT_FILE, map_location=DEVICE))

x_train_embed, y_train_embed = extract_hybrid_features(model, train_loader, device=DEVICE)
x_val_embed, y_val_embed = extract_hybrid_features(model, val_loader, device=DEVICE)
x_test_embed = extract_hybrid_features(model, test_loader, device=DEVICE)

Đang trích xuất đặc trưng...
Đang trích xuất đặc trưng...
Đang trích xuất đặc trưng...


In [21]:
x_train_final = np.concatenate([x_train_embed, x_train_manual], axis=1)
x_val_final = np.concatenate([x_val_embed, x_val_manual], axis=1)
x_test_final = np.concatenate([x_test_embed, x_test_manual], axis=1)

In [22]:
import optuna
from optuna_integration import LightGBMPruningCallback
import lightgbm as lgb
from tqdm.auto import tqdm
import numpy as np

# Tạo sẵn mảng để lưu kết quả dự đoán
ml_val_preds = np.zeros((len(x_val_final), 6), dtype=np.float32)
ml_test_preds = np.zeros((len(x_test_final), 6), dtype=np.float32)

TARGET_INDICES = [1, 2, 3, 5, 6, 7]
N_TRIALS = 30 # Nâng lên 30 để Optuna quét tham số kỹ hơn (nhờ có GPU nên sẽ rất nhanh)

# --- CHUẨN BỊ FULL DATA SẴN SÀNG CHO BƯỚC RETRAIN ---
x_full_train = np.concatenate([x_train_final, x_val_final], axis=0)

def get_optuna_lgb_params(trial):
    return {
        'objective': 'regression',    
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'device_type': 'gpu', # Bật GPU để tăng tốc độ chạy gấp 3-5 lần
        'max_bin': 127,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 128),
        'max_depth': -1, 
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 0.8),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 0.9),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 5),
        'n_jobs': -1,                
        'seed': RANDOM_SEED,
        'verbose': -1,
        'feature_pre_filter': False
    }

for i, target_idx in enumerate(TARGET_INDICES):
    attr_name = ATTRIBUTE_LIST[i]
    
    # 1. Tách nhãn
    y_train_target = y_train_embed[:, target_idx]
    y_val_target = y_val_embed[:, target_idx]
    
    # 2. Gộp nhãn cho Full Data
    y_full_target = np.concatenate([y_train_target, y_val_target], axis=0)
    
    # 3. Tạo các Dataset của LightGBM
    train_data = lgb.Dataset(x_train_final, label=y_train_target, free_raw_data=False)
    val_data = lgb.Dataset(x_val_final, label=y_val_target, reference=train_data, free_raw_data=False)
    full_train_data = lgb.Dataset(x_full_train, label=y_full_target, free_raw_data=False)
    
    # =================================================================
    # KHÂU TỐI ƯU THAM SỐ VỚI OPTUNA (TUYỆT ĐỐI CÔ LẬP VALIDATION)
    # =================================================================
    def objective_lgb(trial):
        params = get_optuna_lgb_params(trial)
        pruning_callback = LightGBMPruningCallback(trial, 'rmse')
        
        gbm_cv = lgb.train(
            params,
            train_data,
            num_boost_round=800,
            valid_sets=[val_data], # Chỉ đánh giá trên tập Val
            callbacks=[
                lgb.early_stopping(stopping_rounds=40, verbose=False),
                pruning_callback
            ]
        )
        return gbm_cv.best_score['valid_0']['rmse']

    study = optuna.create_study(
        direction="minimize", 
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
        pruner=optuna.pruners.HyperbandPruner() 
    )
    
    with tqdm(total=N_TRIALS, desc=f"Tuning Attr_{attr_name}") as pbar:
        def tqdm_callback(study, trial):
            pbar.update(1)
            pbar.set_postfix({"Best RMSE": f"{study.best_value:.4f}"})
            
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study.optimize(objective_lgb, n_trials=N_TRIALS, callbacks=[tqdm_callback])

    best_params = study.best_params
    best_params.update({
        'objective': 'regression', 
        'metric': 'rmse', 
        'boosting_type': 'gbdt', 
        'device_type': 'gpu',
        'max_bin': 127,
        'n_jobs': -1, 
        'seed': RANDOM_SEED, 
        'verbose': -1,
        'feature_pre_filter': False
    })
    
    print(f"Tham số tốt nhất cho Attr_{attr_name}: {best_params}")

    # =================================================================
    # PHASE 1: TÌM SỐ VÒNG LẶP TỐI ƯU (BEST ITERATION) TRÊN TẬP SPLIT
    # =================================================================
    # Bí kíp: Giảm learning_rate đi 1 nửa để mô hình dò điểm dừng chính xác hơn
    best_params['learning_rate'] = best_params['learning_rate'] * 0.5
    
    print(f"-> Đang dò tìm số vòng lặp tối ưu trên tập Split (Train/Val)...")
    search_gbm = lgb.train(
        best_params,
        train_data,
        num_boost_round=3500,
        valid_sets=[train_data, val_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False)
        ]
    )
    optimal_iters = search_gbm.best_iteration
    
    # Lưu kết quả dự đoán Validation để in ra báo cáo WMSE ở cuối
    ml_val_preds[:, i] = search_gbm.predict(x_val_final, num_iteration=optimal_iters)

    # =================================================================
    # PHASE 2: RETRAIN TRÊN FULL DATA VÀ XUẤT DỰ ĐOÁN TẬP TEST
    # =================================================================
    # Tăng thêm 10% số vòng lặp do lượng dữ liệu nạp vào bây giờ đã nhiều hơn (Train + Val)
    retrain_iters = int(optimal_iters * 1.1) 
    print(f"-> Đã khóa optimal_iters = {optimal_iters}. Đang Retrain trên FULL DATA với max {retrain_iters} vòng...")
    
    final_gbm = lgb.train(
        best_params,
        full_train_data,
        num_boost_round=retrain_iters, 
        valid_sets=[full_train_data], # Dùng chính Full Data làm tập valid để chặn Overfit bằng Train Loss
        callbacks=[
            lgb.early_stopping(stopping_rounds=150, verbose=True),
            lgb.log_evaluation(period=500)
        ]
    )
    
    print(f"Hoàn tất Attr_{attr_name} | Retrain Iteration: {final_gbm.best_iteration}\n" + "="*50)
    
    # Dùng Mô hình Retrain mạnh nhất này để dự đoán tập Test
    ml_test_preds[:, i] = final_gbm.predict(x_test_final, num_iteration=final_gbm.best_iteration)

[I 2026-03-18 14:47:08,223] A new study created in memory with name: no-name-031032c9-4420-4b29-a080-c4f680cb0e5e


Tuning Attr_1:   0%|          | 0/30 [00:00<?, ?it/s]

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Tham số tốt nhất cho Attr_1: {'learning_rate': 0.031169409812378812, 'num_leaves': 49, 'min_data_in_leaf': 98, 'reg_alpha': 1.2604664585649468, 'reg_lambda': 5.727904470799623, 'feature_fraction': 0.7579309401710596, 'bagging_fraction': 0.739159991524434, 'bagging_freq': 5, 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'device_type': 'gpu', 'max_bin': 127, 'n_jobs': -1, 'seed': 42, 'verbose': -1, 'feature_pre_filter': False}
-> Đang dò tìm số vòng lặp tối ưu trên tập Split (Train/Val)...
-> Đã khóa optimal_iters = 292. Đang Retrain trên FULL DATA với max 321 vòng...
Hoàn tất Attr_1 | Retrain Iteration: 0


Tuning Attr_2:   0%|          | 0/30 [00:00<?, ?it/s]

...

Tham số tốt nhất cho Attr_4: {'learning_rate': 0.011448469784568732, 'num_leaves': 123, 'min_data_in_leaf': 98, 'reg_alpha': 1.7123375973163988, 'reg_lambda': 0.01653693718282442, 'feature_fraction': 0.43906884560255355, 'bagging_fraction': 0.7736932106048628, 'bagging_freq': 3, 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'device_type': 'gpu', 'max_bin': 127, 'n_jobs': -1, 'seed': 42, 'verbose': -1, 'feature_pre_filter': False}
-> Đang dò tìm số vòng lặp tối ưu trên tập Split (Train/Val)...
-> Đã khóa optimal_iters = 781. Đang Retrain trên FULL DATA với max 859 vòng...
[500]	training's rmse: 1.0481
Hoàn tất Attr_4 | Retrain Iteration: 0


Tuning Attr_5:   0%|          | 0/30 [00:00<?, ?it/s]

Tham số tốt nhất cho Attr_5: {'learning_rate': 0.021789307659775742, 'num_leaves': 124, 'min_data_in_leaf': 79, 'reg_alpha': 0.24810409748678125, 'reg_lambda': 0.004207988669606638, 'feature_fraction': 0.46239780813448106, 'bagging_fraction': 0.5232334448672797, 'bagging_freq': 5, 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'device_type': 'gpu', 'max_bin': 127, 'n_jobs': -1, 'seed': 42, 'verbose': -1, 'feature_pre_filter': False}
-> Đang dò tìm số vòng lặp tối ưu trên tập Split (Train/Val)...
-> Đã khóa optimal_iters = 395. Đang Retrain trên FULL DATA với max 434 vòng...
Hoàn tất Attr_5 | Retrain Iteration: 0


Tuning Attr_6:   0%|          | 0/30 [00:00<?, ?it/s]

Tham số tốt nhất cho Attr_6: {'learning_rate': 0.034902854606300035, 'num_leaves': 100, 'min_data_in_leaf': 21, 'reg_alpha': 7.579479953348009, 'reg_lambda': 2.1368329072358767, 'feature_fraction': 0.4849356442713105, 'bagging_fraction': 0.5727299868828403, 'bagging_freq': 1, 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'device_type': 'gpu', 'max_bin': 127, 'n_jobs': -1, 'seed': 42, 'verbose': -1, 'feature_pre_filter': False}
-> Đang dò tìm số vòng lặp tối ưu trên tập Split (Train/Val)...
-> Đã khóa optimal_iters = 327. Đang Retrain trên FULL DATA với max 359 vòng...
Hoàn tất Attr_6 | Retrain Iteration: 0


In [23]:
def evaluate_wmse(y_true, y_pred):
    M, W = np.array(M_LIST_METRIC), np.array(W_LIST_VALS)
    return np.mean(np.sum(W * np.power((y_true - y_pred) / M, 2), axis=1))
    
def wmape(y_t, y_p):
    sum_abs_error = np.sum(np.abs(y_t - y_p))
    sum_abs_true = np.sum(np.abs(y_t))
    return (sum_abs_error / (sum_abs_true + 1e-8)) * 100

print("\n" + "=" * 76)
print(f"{'THUỘC TÍNH':<12} | {'MAE':<10} | {'MSE':<12} | {'RMSE':<10} | {'WMAPE':<10}")
print("-" * 76)

ml_val_tensor = torch.tensor(ml_val_preds, device=DEVICE, dtype=torch.float32)
val_preds = post_process_predictions(ml_val_tensor).cpu().numpy()

for i, attr in enumerate(ATTRIBUTE_LIST):
    y_t = y_val_orig[:, i]
    y_p = val_preds[:, i]
    
    mae = np.mean(np.abs(y_t - y_p))
    mse = np.mean(np.square(y_t - y_p))
    rmse = np.sqrt(mse)
    wmape_val = wmape(y_t, y_p)
    
    print(f"Attr {attr:<7} | {mae:<10.4f} | {mse:<12.4f} | {rmse:<10.4f} | {wmape_val:<10.4f}")

print("=" * 76)

overall_mae = np.mean(np.abs(y_val_orig - val_preds))
overall_mse = np.mean(np.square(y_val_orig - val_preds))
overall_rmse = np.sqrt(overall_mse)
overall_wmape = wmape(y_val_orig, val_preds)
final_wmse = evaluate_wmse(y_val_orig, val_preds)

print(f"{'MAE':<15}: {overall_mae:.4f}")
print(f"{'MSE':<15}: {overall_mse:.4f}")
print(f"{'RMSE':<15}: {overall_rmse:.4f}")
print(f"{'WMAPE':<15}: {overall_wmape:.4f}")
print(f"{'WMSE':<15}: {final_wmse:.4f}")
print("=" * 76)


THUỘC TÍNH   | MAE        | MSE          | RMSE       | WMAPE     
----------------------------------------------------------------------------
Attr 1       | 0.4187     | 1.5489       | 1.2446     | 6.2524    
Attr 2       | 1.0352     | 7.9805       | 2.8250     | 8.1213    
Attr 3       | 0.6078     | 12.9207      | 3.5945     | 1.2231    
Attr 4       | 0.5372     | 2.1442       | 1.4643     | 8.0266    
Attr 5       | 1.6019     | 14.3457      | 3.7876     | 11.9975   
Attr 6       | 0.6390     | 11.6212      | 3.4090     | 1.2838    
MAE            : 0.8066
MSE            : 8.4269
RMSE           : 2.9029
WMAPE          : 3.4831
WMSE           : 0.2993


In [24]:
ml_tensor = torch.tensor(ml_test_preds, device=DEVICE, dtype=torch.float32)
processed_ml_preds = post_process_predictions(ml_tensor)

test_preds = processed_ml_preds.cpu().numpy()
submission_df = pd.DataFrame({"id": x_test["id"]})

for idx, attr in enumerate(ATTRIBUTE_LIST):
    submission_df[f"attr_{attr}"] = test_preds[:, idx].astype(np.uint16)

submission_df.to_csv(SUBMISSION_FILE, index=False)
print(submission_df)
print(submission_df.dtypes)

           id  attr_1  attr_2  attr_3  attr_4  attr_5  attr_6
0       n6r61       2      11      48       9      23      73
1       jkctd       5      15      63      10      18      67
2       a4060       2       8      39       4       5      72
3       fkbud       9      20       4       5       7      18
4       ui5gu       8      26      26       7      16      56
...       ...     ...     ...     ...     ...     ...     ...
108668  phk1w       7      12      63       7      17      11
108669  sgb1f       6      14      90       6      21      94
108670  e3s26       7      14      61       8      16      94
108671  d0kwv       6      22      40      10      18      50
108672  xfvdy       3      23      43      12      25      17

[108673 rows x 7 columns]
id        object
attr_1    uint16
attr_2    uint16
attr_3    uint16
attr_4    uint16
attr_5    uint16
attr_6    uint16
dtype: object
